In [13]:
import pandas as pd

In [20]:
df=pd.read_csv(r"C:\Users\kadir\OneDrive\Masaüstü\korelaktal\korelaktal_hasta_hikayeleri.csv")
print(df.head())

   Label                                       Hasta_Yorumu
0      0  Kontrol amaçlı yazıyorum. 77 yaşında bir erkeğ...
1      0  İyi günler. Yaşım 59, cinsiyetim erkek. Birinc...
2      1  Kontrol amaçlı yazıyorum. 66 yaşında bir erkeğ...
3      1  Kontrol amaçlı yazıyorum. 83 yaşında bir erkeğ...
4      0  Doktor bey/hanım merhabalar, 66 yaşındayım. Bi...


In [21]:
import pandas as pd
import re
import string
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

def metin_temizle(metin):
    if pd.isna(metin): return ""
    metin = str(metin).lower()
    noktalama = re.escape(string.punctuation)
    metin = re.sub(rf'[{noktalama}]', ' ', metin)
    return re.sub(r'\s+', ' ', metin).strip()

# --- BİLİMSEL VE OLASILIKSAL ETİKETLEME FONKSİYONU ---
def bilimsel_risk_hesapla(metin):
    m = str(metin).lower()
    
    # Her insanın temel bir kanser riski vardır (örneğin genel popülasyonda %2)
    olasilik = 0.02 
    
    # Risk faktörleri bu olasılığı gerçekçi oranlarda artırır
    if "ailemde" in m and "kanser" in m: olasilik += 0.15 # %15 ek risk
    if "iltihabi" in m: olasilik += 0.10
    if "obez" in m or "ciddi kilo" in m: olasilik += 0.06
    if "sigara" in m: olasilik += 0.05
    if "alkol" in m: olasilik += 0.03
    
    yas_bul = re.search(r'(\d+)\s+yaş', m)
    if yas_bul:
        yas = int(yas_bul.group(1))
        if yas > 50: olasilik += 0.08
        if yas > 65: olasilik += 0.10
        
    # Olasılık maksimum %95 olabilir (Tıpta %100 kesinlik yoktur)
    olasilik = min(olasilik, 0.95)
    
    # GERÇEK HAYAT SİMÜLASYONU: 
    # Numpy, hesaplanan bu olasılığa göre ağırlıklı yazı-tura atar.
    # Örneğin olasılık 0.30 ise, %30 ihtimalle 1 (Kanser), %70 ihtimalle 0 (Sağlıklı) döner.
    return np.random.choice([0, 1], p=[1-olasilik, olasilik])

print("1. Bilimsel ve olasılıksal etiketler (Labeller) hesaplanıyor...")
df['Temiz_Metin'] = df['Hasta_Yorumu'].apply(metin_temizle)
df['Bilimsel_Label'] = df['Hasta_Yorumu'].apply(bilimsel_risk_hesapla)

print("2. Metinler matematiğe (TF-IDF) çevriliyor...")
X = df['Temiz_Metin']
y = df['Bilimsel_Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vektor = vectorizer.fit_transform(X_train)
X_test_vektor = vectorizer.transform(X_test)

negatif_sayisi = sum(y_train == 0)
pozitif_sayisi = sum(y_train == 1)
denge_orani = negatif_sayisi / pozitif_sayisi

print("3. XGBoost Modeli gerçekçi belirsizliklerle eğitiliyor...")
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    learning_rate=0.05, # Öğrenme hızını biraz düşürdük, daha ince öğrensin
    max_depth=5,        # Aşırı ezberlemeyi önlemek için derinliği azalttık
    n_estimators=300,
    scale_pos_weight=denge_orani,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train_vektor, y_train)

tahminler = xgb_model.predict(X_test_vektor)
dogruluk = accuracy_score(y_test, tahminler)

print(f"\n*** BİLİMSEL SİMÜLASYON: Doğruluk Oranı (Accuracy): % {dogruluk * 100:.2f} ***\n")
print("--- Detaylı Başarı Raporu ---")
print(classification_report(y_test, tahminler))

1. Bilimsel ve olasılıksal etiketler (Labeller) hesaplanıyor...
2. Metinler matematiğe (TF-IDF) çevriliyor...
3. XGBoost Modeli gerçekçi belirsizliklerle eğitiliyor...

*** BİLİMSEL SİMÜLASYON: Doğruluk Oranı (Accuracy): % 61.94 ***

--- Detaylı Başarı Raporu ---
              precision    recall  f1-score   support

           0       0.81      0.63      0.71     24481
           1       0.37      0.59      0.45      9019

    accuracy                           0.62     33500
   macro avg       0.59      0.61      0.58     33500
weighted avg       0.69      0.62      0.64     33500



In [15]:
# Metin temizleme adımları
import pandas as pd
import re
import string

def clean_text(text):
    # 1. Küçük harfe çevirme
    text = text.lower()
    
    noktalama_isaretleri = re.escape(string.punctuation)
    text = re.sub(rf'[{noktalama_isaretleri}]', ' ', text)
    # 4. Gereksiz boşlukların kaldırılması
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['Temiz_Metin'] = df['Hasta_Yorumu'].apply(clean_text)

# 4. Temizlenmiş verinin ilk 3 satırına bakalım
print(df[['Hasta_Yorumu', 'Temiz_Metin']].head(3))


                                        Hasta_Yorumu  \
0  Kontrol amaçlı yazıyorum. 77 yaşında bir erkeğ...   
1  İyi günler. Yaşım 59, cinsiyetim erkek. Birinc...   
2  Kontrol amaçlı yazıyorum. 66 yaşında bir erkeğ...   

                                         Temiz_Metin  
0  kontrol amaçlı yazıyorum 77 yaşında bir erkeği...  
1  i̇yi günler yaşım 59 cinsiyetim erkek birinci ...  
2  kontrol amaçlı yazıyorum 66 yaşında bir erkeği...  


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df['Temiz_Metin'] 
y = df['Label']       

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Eğitim için ayrılan satır sayısı: {len(X_train)}")
print(f"Test için ayrılan satır sayısı: {len(X_test)}\n")

# 3. Tokenizasyon ve Metin Temsili (TF-IDF)
# max_features=5000: Model çok fazla gereksiz kelimeyle yorulmasın diye, 
# veri setinde en çok geçen/en önemli 5000 kelimeyi seçmesini söylüyoruz.
vectorizer = TfidfVectorizer(max_features=5000)

# Eğitim verisiyle sözlüğü oluştur ve sayılara çevir (FIT ve TRANSFORM)
X_train_vektor = vectorizer.fit_transform(X_train)

# Test verisini SADECE sayılara çevir (Kopya çekmemesi için sadece TRANSFORM yapıyoruz, FIT yapmıyoruz)
X_test_vektor = vectorizer.transform(X_test)

# Sonucu Görelim
print("Tokenizasyon ve Vektörizasyon Başarılı!")
print(f"Eğitim Verisi Matris Boyutu: {X_train_vektor.shape}")

Eğitim için ayrılan satır sayısı: 133997
Test için ayrılan satır sayısı: 33500

Tokenizasyon ve Vektörizasyon Başarılı!
Eğitim Verisi Matris Boyutu: (133997, 187)


In [17]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# 1. Sınıf dengesizliği oranını otomatik hesaplama
negatif_sayisi = sum(y_train == 0)
pozitif_sayisi = sum(y_train == 1)
denge_orani = negatif_sayisi / pozitif_sayisi

# 2. Sağlam ve Dengeli XGBoost Modeli
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    learning_rate=0.1,
    max_depth=6,
    n_estimators=200,
    scale_pos_weight=denge_orani,  # Sınıfları dengeleyen kritik parametre
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# 3. Modeli Eğitme ve Test Etme
print("XGBoost modeli dengeli ayarlarla eğitiliyor...")
xgb_model.fit(X_train_vektor, y_train)

tahminler = xgb_model.predict(X_test_vektor)
dogruluk = accuracy_score(y_test, tahminler)

print(f"\n*** YENİ XGBoost Modelinin Doğruluk Oranı (Accuracy): % {dogruluk * 100:.2f} ***\n")
print("--- XGBoost Detaylı Başarı Raporu ---")
print(classification_report(y_test, tahminler))

XGBoost modeli dengeli ayarlarla eğitiliyor...

*** YENİ XGBoost Modelinin Doğruluk Oranı (Accuracy): % 50.07 ***

--- XGBoost Detaylı Başarı Raporu ---
              precision    recall  f1-score   support

           0       0.40      0.49      0.44     13419
           1       0.60      0.51      0.55     20081

    accuracy                           0.50     33500
   macro avg       0.50      0.50      0.49     33500
weighted avg       0.52      0.50      0.51     33500



In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Modeli Tanımlama
# max_iter=1000: Modelin matematiksel denklemi çözerken pes etmeden önce yapacağı maksimum deneme sayısı
model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')

print("Yapay zeka modeli eğitiliyor, lütfen bekleyin...")
# 2. Modeli Eğitme (Öğrenme aşaması)
model.fit(X_train_vektor, y_train)
print("Model eğitimi başarıyla tamamlandı!\n")

# 3. Test Verisiyle Sınava Sokma
# Bakalım daha önce HİÇ GÖRMEDİĞİ 33.500 hastayı ne kadar doğru tahmin edecek?
tahminler = model.predict(X_test_vektor)

# 4. Doğruluk (Accuracy) Oranını Hesaplama
dogruluk = accuracy_score(y_test, tahminler)

print(f"*** Modelin Doğruluk Oranı (Accuracy): % {dogruluk * 100:.2f} ***\n")

# 5. Detaylı Karne (Hangi sınıfta daha başarılı?)
print("--- Detaylı Başarı Raporu ---")
print(classification_report(y_test, tahminler))

Yapay zeka modeli eğitiliyor, lütfen bekleyin...
Model eğitimi başarıyla tamamlandı!

*** Modelin Doğruluk Oranı (Accuracy): % 49.80 ***

--- Detaylı Başarı Raporu ---
              precision    recall  f1-score   support

           0       0.40      0.51      0.45     13419
           1       0.60      0.49      0.54     20081

    accuracy                           0.50     33500
   macro avg       0.50      0.50      0.49     33500
weighted avg       0.52      0.50      0.50     33500



In [ ]:
# Rakamları ve noktalama işaretlerini silen kod
import re

# Örnek bir metin sütunu içeren DataFrame varsayıyoruz
data = {'metin': ['Bu bir örnek metindir! 123', 'Python programlama dili çok güçlüdür.', '123 numaralı örnek veri.']}
df = pd.DataFrame(data)

# Rakamları ve noktalama işaretlerini kaldırma
df['metin'] = df['metin'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', x))

# Temizlenmiş metinleri görüntüleme
print(df)